# 80/20 BBB Model Training Trace
This notebook keeps the training trace for the pretrained `../output/models/hypertuning/*_8020_best_model.pkl` models. It is not required for the pretrained trial/figure workflow.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, matthews_corrcoef, confusion_matrix
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

try:
    from imblearn.pipeline import Pipeline
    from imblearn.under_sampling import RandomUnderSampler
except Exception:
    Pipeline = SkPipeline
    RandomUnderSampler = None

try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False

try:
    from lightgbm import LGBMClassifier
    LGBM_AVAILABLE = True
except Exception:
    LGBM_AVAILABLE = False

RANDOM_SEED = 42
OUTPUT_DIR = Path('../output/models/hypertuning')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Load 80/20 Split

In [ ]:
def load_split(split_ratio='8020', base_dir='../data/split'):
    base = Path(base_dir) / split_ratio
    X_train = pd.read_csv(base / 'x_train.csv', index_col=0).select_dtypes(include=[np.number])
    y_train = pd.read_csv(base / 'y_train.csv', index_col=0).iloc[:, 0].astype(int)
    X_test = pd.read_csv(base / 'x_test.csv', index_col=0)[X_train.columns].select_dtypes(include=[np.number])
    y_test = pd.read_csv(base / 'y_test.csv', index_col=0).iloc[:, 0].astype(int)
    return X_train.replace([np.inf, -np.inf], np.nan), y_train, X_test.replace([np.inf, -np.inf], np.nan), y_test

X_train, y_train, X_test, y_test = load_split('8020')
print(f'Train: {X_train.shape} | positives={int((y_train == 1).sum())} negatives={int((y_train == 0).sum())}')
print(f'Test : {X_test.shape} | positives={int((y_test == 1).sum())} negatives={int((y_test == 0).sum())}')


## Model Pipelines and GridSearchCV Spaces

In [ ]:
def build_pipeline(model):
    steps = [
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
    if RandomUnderSampler is not None:
        steps.append(('undersampler', RandomUnderSampler(random_state=RANDOM_SEED)))
    steps.append(('clf', model))
    return Pipeline(steps)


def get_models():
    models = {
        'LogReg': LogisticRegression(max_iter=5000, random_state=RANDOM_SEED),
        'KNN': KNeighborsClassifier(),
        'RF': RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=-1),
        'ET': ExtraTreesClassifier(random_state=RANDOM_SEED, n_jobs=-1),
    }
    if XGB_AVAILABLE:
        models['XGB'] = XGBClassifier(random_state=RANDOM_SEED, eval_metric='logloss', verbosity=0, n_jobs=-1)
    if LGBM_AVAILABLE:
        models['LGBM'] = LGBMClassifier(random_state=RANDOM_SEED, verbose=-1, n_jobs=-1)
    return models

param_grids = {
    'LogReg': {'clf__C': [0.1, 1, 10], 'clf__penalty': ['l2'], 'clf__solver': ['lbfgs']},
    'KNN': {'clf__n_neighbors': [5, 9, 15], 'clf__weights': ['distance'], 'clf__metric': ['euclidean', 'manhattan']},
    'RF': {'clf__n_estimators': [100, 200], 'clf__max_depth': [10, 20], 'clf__min_samples_split': [2, 10], 'clf__bootstrap': [True]},
    'ET': {'clf__n_estimators': [100, 200], 'clf__max_depth': [10, 20], 'clf__min_samples_split': [2, 10]},
    'XGB': {'clf__n_estimators': [100, 200], 'clf__max_depth': [3, 6], 'clf__learning_rate': [0.05, 0.1], 'clf__subsample': [1.0]},
    'LGBM': {'clf__n_estimators': [100, 200], 'clf__max_depth': [4, 6], 'clf__learning_rate': [0.05, 0.1], 'clf__num_leaves': [31, 50]},
}


## Train, Tune, Evaluate, and Save Models

In [ ]:
def predict_scores(model, X):
    y_pred = model.predict(X)
    if hasattr(model, 'predict_proba'):
        y_score = model.predict_proba(X)[:, 1]
    elif hasattr(model, 'decision_function'):
        raw = model.decision_function(X)
        y_score = (raw - raw.min()) / (raw.max() - raw.min() + 1e-12)
    else:
        y_score = y_pred.astype(float)
    return y_pred, y_score


def compute_metrics(y_true, y_pred, y_score):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_score),
        'mcc': matthews_corrcoef(y_true, y_pred),
        'sensitivity': recall_score(y_true, y_pred, zero_division=0),
        'specificity': tn / (tn + fp) if (tn + fp) else np.nan,
    }

models = get_models()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
results = []

for model_name, model in models.items():
    print('\n' + '=' * 80)
    print(f'Training {model_name}')
    pipeline = build_pipeline(model)
    search = GridSearchCV(
        pipeline,
        param_grids[model_name],
        scoring='balanced_accuracy',
        cv=cv,
        n_jobs=1,
        refit=True,
        verbose=1,
    )
    search.fit(X_train, y_train)
    best_model = search.best_estimator_
    y_pred, y_score = predict_scores(best_model, X_test)
    metrics = compute_metrics(y_test, y_pred, y_score)

    row = {
        'model': model_name,
        'split': '8020',
        'balance_method': 'undersampling' if RandomUnderSampler is not None else 'none',
        'best_cv_score': search.best_score_,
        'best_params': str(search.best_params_),
        **metrics,
    }
    results.append(row)

    joblib.dump(best_model, OUTPUT_DIR / f'{model_name}_8020_best_model.pkl')
    pd.DataFrame({'y_true': y_test.values, 'y_pred': y_pred, 'y_score': y_score}).to_csv(
        OUTPUT_DIR / f'predictions_{model_name}_8020.csv', index=False
    )
    print(f"Best CV balanced accuracy: {search.best_score_:.4f}")
    print(f"Test Acc={metrics['accuracy']:.4f} | BalAcc={metrics['balanced_accuracy']:.4f} | F1={metrics['f1']:.4f} | ROC-AUC={metrics['roc_auc']:.4f}")

results_df = pd.DataFrame(results).sort_values('roc_auc', ascending=False)
results_df.to_csv(OUTPUT_DIR / 'results_8020_hypertuned.csv', index=False)
display(results_df)
